# 07 — Live Demo
**Home Credit Default Risk — Presentation Support Notebook**

Self-contained: loads the saved production pipeline, picks a real raw applicant from the test set, shows the prediction, and generates a SHAP waterfall plot explaining that one decision — exactly what the 'Live Demo' section of the presentation walks through.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.model_selection import train_test_split

DATA_DIR = '../data/'

# Load the SAME raw data and split used to build the production pipeline (06)
raw = pd.read_csv(DATA_DIR + 'application_train.csv')
raw['DAYS_EMPLOYED'] = raw['DAYS_EMPLOYED'].replace(365243, np.nan)

X_raw = raw.drop(columns=['TARGET', 'SK_ID_CURR'])
y_raw = raw['TARGET']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=42
)

# Load the saved production pipeline
pipeline = joblib.load('../models/credit_scoring_pipeline.pkl')
print("Pipeline loaded:", type(pipeline.named_steps['classifier']).__name__)


## 1. Pick a Real High-Risk Applicant from the Test Set

No invented numbers — this is a genuine raw applicant row the model has never seen during training.

In [ ]:
all_proba = pipeline.predict_proba(X_test_raw)[:, 1]
proba_series = pd.Series(all_proba, index=X_test_raw.index)

# Pick a genuine high-risk applicant (PD > 0.5); fall back to the highest-scored one if none exist
high_risk_candidates = proba_series[proba_series > 0.5]
demo_idx = high_risk_candidates.index[0] if len(high_risk_candidates) > 0 else proba_series.idxmax()

demo_row = X_test_raw.loc[[demo_idx]]
demo_pd = proba_series[demo_idx]
demo_actual = y_test_raw.loc[demo_idx]

print(f"Selected applicant — index {demo_idx}")
print(f"Predicted probability of default: {demo_pd:.4f}")
print(f"Actual outcome (TARGET): {demo_actual}  (1 = defaulted, 0 = repaid)")


## 2. Run the Prediction — This Is the "Live" Part

Call the saved pipeline directly on the raw row — no manual preprocessing.

In [ ]:
prediction_proba = pipeline.predict_proba(demo_row)[:, 1][0]
decision = "DECLINE (high risk)" if prediction_proba > 0.20 else "APPROVE (low risk)"

print(f"pipeline.predict_proba(raw_row) -> {prediction_proba:.4f}")
print(f"Decision: {decision}")


## 3. Explain the Decision — SHAP Waterfall for This One Applicant

In [ ]:
# Transform the raw row through the pipeline's preprocessor to get the actual
# feature matrix the classifier sees, and the real feature names after encoding.
preprocessor = pipeline.named_steps['preprocessor']
classifier = pipeline.named_steps['classifier']

demo_transformed = preprocessor.transform(demo_row)
feature_names = preprocessor.get_feature_names_out()

# TreeExplainer on the classifier step directly (fast, exact for tree-based models)
explainer = shap.TreeExplainer(classifier)
sv = explainer.shap_values(demo_transformed)

if isinstance(sv, list):
    sv_pos = sv[1][0]
    base_val = explainer.expected_value[1]
else:
    sv_pos = sv[0]
    base_val = explainer.expected_value

explanation = shap.Explanation(
    values=sv_pos,
    base_values=base_val,
    data=demo_transformed[0] if not hasattr(demo_transformed, 'toarray') else demo_transformed.toarray()[0],
    feature_names=feature_names
)

plt.figure()
shap.waterfall_plot(explanation, max_display=10, show=False)
plt.title(f'Applicant {demo_idx} — Predicted PD = {prediction_proba:.3f}')
plt.tight_layout()
plt.savefig('../figs/live_demo_waterfall.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Top Factors — Adverse Action Language

In [ ]:
contributions = pd.Series(sv_pos, index=feature_names).sort_values(ascending=False)
top4 = contributions.head(4)

print(f"Top 4 factors contributing to elevated risk for applicant {demo_idx}:\n")
for i, (feat, impact) in enumerate(top4.items(), 1):
    print(f"{i}. {feat}  (SHAP contribution: {impact:.4f})")


**Interpretation:** *(Translate the top 4 raw feature names above into plain adverse-action language during the live walkthrough — e.g. "External bureau score was below threshold," "Prior payment delinquency on file" — the same style used in the Step 5 adverse action notice.)*

**Backup note:** if live execution isn't possible during the actual presentation (Wi-Fi, projector setup, etc.), the saved figure `../figs/live_demo_waterfall.png` and the printed values above can be shown as a pre-run screenshot instead — the code itself is the proof it's genuinely reproducible, not just a static image.